In [4]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [5]:
from src.configs import CELL_SIZE, IMAGE_SIZE
from torch import round as rd

def convert_xywh_coords(bbox: tuple, row, col, draw: bool):
    x = (bbox[0] * CELL_SIZE) + (col * CELL_SIZE)
    y = (bbox[1] * CELL_SIZE) + (row * CELL_SIZE)
    w = bbox[2] * IMAGE_SIZE
    h = bbox[3] * IMAGE_SIZE
    
    xmin = x - w/2
    ymin = y - h/2
    xmax = x + w/2
    ymax = y + h/2

    if draw:
        return (int(rd(xmin)), int(rd(ymin)), int(rd(xmax)), int(rd(ymax)))        
    return (xmin, ymin, xmax, ymax)

In [6]:
import torch

def area(bbox: tuple):
    w = torch.clamp(bbox[2] - bbox[0], min=0)
    h = torch.clamp(bbox[3] - bbox[1], min=0)
    return w * h

In [7]:
tensor_1 = torch.tensor((0.125, 0.7812, 0.0714, 0.0893))
tensor_2 = torch.tensor((0.133, 0.7243, 0.0945, 0.0922))

print(convert_xywh_coords(tensor_1, 3, 4, False))
print(convert_xywh_coords(tensor_1, 3, 4, True))

(tensor(124.0032), tensor(110.9968), tensor(139.9968), tensor(131.))
(124, 111, 140, 131)


In [13]:
def IoU(target_bbox: tuple, pred_bbox: tuple, row, col):
    # 1. convert (x, y, w, h) to (xmin, ymin, xmax, ymax)
    target = convert_xywh_coords(target_bbox, row, col, False)
    pred = convert_xywh_coords(pred_bbox, row, col, False)

    # 2. find intersection box coordinates
    xmin = torch.max(target[0], pred[0])
    ymin = torch.max(target[1], pred[1])
    xmax = torch.min(target[2], pred[2])
    ymax = torch.min(target[3], pred[3])

    inter = (xmin, ymin, xmax, ymax)

    print(target[0], pred[0], xmin)

    # 3. find areas of boxes
    target_area = area(target)
    pred_area = area(pred)
    inter_area = area(inter)
    union_area = target_area + pred_area - inter_area

    # 4. compute IoU
    return inter_area / union_area if union_area != 0 else 0

In [14]:
IoU(tensor_1, tensor_2, 3, 4)

tensor(124.0032) tensor(121.6720) tensor(124.0032)


tensor(0.6419)

In [26]:
target = torch.zeros(7, 7, 30)
target.shape

torch.Size([7, 7, 30])

In [27]:
target = target.flatten(0, 1)
target.shape

torch.Size([49, 30])

In [31]:
target[0][20:24]

tensor([0., 0., 0., 0.])